# Streamlit Movie Recommendation System

## Objective

In this notebook, we build an interactive Movie Recommendation System using Streamlit.

The application allows users to:

- Select a User ID
- Choose the recommendation model
- Generate personalized movie recommendations
- Display predicted ratings
- Provide an easy-to-use interface

This notebook serves as the prototype before creating the final `app.py`.

In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import torch

print("Libraries Loaded Successfully!")

Libraries Loaded Successfully!


# Load MovieLens Dataset

In [2]:
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id","movie_id","rating","timestamp"]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0,1],
    names=["movie_id","movie_title"]
)

ratings = ratings.merge(
    movies,
    on="movie_id"
)

print("Dataset Loaded Successfully!")

ratings.head()

Dataset Loaded Successfully!


,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


# Create User & Movie Encoding

In [3]:
user_encoder = {
    user:i
    for i,user in enumerate(
        sorted(ratings.user_id.unique())
    )
}

movie_encoder = {
    movie:i
    for i,movie in enumerate(
        sorted(ratings.movie_id.unique())
    )
}

movie_decoder = {
    v:k
    for k,v in movie_encoder.items()
}

print("Encoding Created Successfully!")

print("Users :",len(user_encoder))
print("Movies:",len(movie_encoder))

Encoding Created Successfully!
Users : 943
Movies: 1682


# Load Trained Neural Collaborative Filtering Model

In [4]:
class NCF(torch.nn.Module):

    def __init__(self,
                 num_users,
                 num_movies,
                 embedding_dim=50):

        super().__init__()

        self.user_embedding = torch.nn.Embedding(
            num_users,
            embedding_dim
        )

        self.movie_embedding = torch.nn.Embedding(
            num_movies,
            embedding_dim
        )

        self.fc1 = torch.nn.Linear(
            embedding_dim*2,
            128
        )

        self.fc2 = torch.nn.Linear(
            128,
            64
        )

        self.output = torch.nn.Linear(
            64,
            1
        )

        self.relu = torch.nn.ReLU()

        self.dropout = torch.nn.Dropout(0.2)

    def forward(self,user,movie):

        user_vec = self.user_embedding(user)
        movie_vec = self.movie_embedding(movie)

        x = torch.cat(
            [user_vec,movie_vec],
            dim=1
        )

        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))

        return self.output(x)

In [5]:

model = NCF(
    ratings.user_id.nunique(),
    ratings.movie_id.nunique()
)

model.load_state_dict(
    torch.load(
        "models/ncf_model.pth",
        map_location="cpu"
    )
)

model.eval()

print("NCF Model Loaded Successfully!")



NCF Model Loaded Successfully!


# Create Recommendation Function

This function generates Top-N movie recommendations for a selected user using the trained NCF model.

In [6]:
def recommend_movies(user_id, top_n=10):

    encoded_user = user_encoder[user_id]

    watched_movies = ratings[
        ratings.user_id == user_id
    ]["movie_id"].tolist()

    unseen_movies = [
        movie
        for movie in movie_encoder.keys()
        if movie not in watched_movies
    ]

    predictions = []

    for movie in unseen_movies:

        user_tensor = torch.LongTensor([encoded_user])

        movie_tensor = torch.LongTensor(
            [movie_encoder[movie]]
        )

        with torch.no_grad():

            prediction = model(
                user_tensor,
                movie_tensor
            ).item()

        predictions.append(
            (movie,prediction)
        )

    predictions.sort(
        key=lambda x:x[1],
        reverse=True
    )

    top_movies = predictions[:top_n]

    recommendation_df = pd.DataFrame(
        top_movies,
        columns=[
            "movie_id",
            "predicted_rating"
        ]
    )

    recommendation_df = recommendation_df.merge(
        movies,
        on="movie_id"
    )

    return recommendation_df

# Movie Recommendation App using Streamlit

In this notebook, we build a complete Movie Recommendation Web Application using Streamlit.

The application allows a user to:

- Select a User ID
- Generate Top Movie Recommendations
- View predicted ratings
- Display recommendations in a clean interface

This notebook integrates the trained Neural Collaborative Filtering (NCF) model into an interactive web application.

In [7]:
%%writefile app.py

UsageError: %%writefile is a cell magic, but the cell body is empty.


In [8]:
import streamlit as st
import pandas as pd
import numpy as np
import torch

from torch import nn

In [9]:
# ---------------------------------
# Page Configuration
# ---------------------------------

st.set_page_config(
    page_title="Movie Recommendation System",
    layout="wide"
)

st.title("🎬 Movie Recommendation System")

st.write(
    "Neural Collaborative Filtering based Recommendation Engine"
)

2026-07-22 11:15:06.111 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 11:15:06.112 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 11:15:06.125 
  command:

    streamlit run /home/guest/anaconda3/envs/sarvesh/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-07-22 11:15:06.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 11:15:06.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 11:15:06.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 11:15:06.125 Thread 'MainThread': missing ScriptRunContext! This warning can

In [10]:
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id","movie_id","rating","timestamp"]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0,1]
)

movies.columns = [
    "movie_id",
    "movie_title"
]

ratings = ratings.merge(
    movies,
    on="movie_id"
)

st.success("Dataset Loaded Successfully")

2026-07-22 11:15:21.898 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 11:15:21.898 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-22 11:15:21.898 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [11]:
user_encoder = {
    x:i for i,x in enumerate(
        ratings.user_id.unique()
    )
}

movie_encoder = {
    x:i for i,x in enumerate(
        ratings.movie_id.unique()
    )
}

movie_decoder = {
    v:k for k,v in movie_encoder.items()
}

In [12]:
class NCF(nn.Module):

    def __init__(self,n_users,n_movies,embedding_dim=50):

        super().__init__()

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.movie_embedding = nn.Embedding(
            n_movies,
            embedding_dim
        )

        self.fc1 = nn.Linear(
            embedding_dim*2,
            128
        )

        self.fc2 = nn.Linear(
            128,
            64
        )

        self.output = nn.Linear(
            64,
            1
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(0.2)

    def forward(self,user,movie):

        u = self.user_embedding(user)

        m = self.movie_embedding(movie)

        x = torch.cat([u,m],dim=1)

        x = self.relu(self.fc1(x))

        x = self.dropout(x)

        x = self.relu(self.fc2(x))

        return self.output(x)

# Load Trained Model Weights

In this section, we load the trained Neural Collaborative Filtering (NCF) model from disk and switch it to evaluation mode.

This model will be used to predict ratings for unseen movies.

In [13]:
model = NCF(
    len(user2encoded),
    len(movie2encoded)
)

model.load_state_dict(
    torch.load(
        "models/ncf_model.pth",
        map_location=torch.device("cpu")
    )
)

model.eval()

print("NCF Model Loaded Successfully!")

NameError: name 'user2encoded' is not defined

In [14]:
print([x for x in globals().keys() if "user" in x.lower()])

['user_encoder']


In [15]:
print([x for x in globals().keys() if "movie" in x.lower()])

['movies', 'movie_encoder', 'movie_decoder', 'recommend_movies']


In [16]:
user2encoded = user_encoder
movie2encoded = movie_encoder

encoded2user = {v: k for k, v in user2encoded.items()}
encoded2movie = {v: k for k, v in movie2encoded.items()}

In [17]:
# User Encoding
user2encoded = {
    user: idx
    for idx, user in enumerate(ratings["user_id"].unique())
}

# Movie Encoding
movie2encoded = {
    movie: idx
    for idx, movie in enumerate(ratings["movie_id"].unique())
}

# Reverse Mappings
encoded2user = {
    idx: user
    for user, idx in user2encoded.items()
}

encoded2movie = {
    idx: movie
    for movie, idx in movie2encoded.items()
}

print("Encoding Created Successfully!")
print("Users :", len(user2encoded))
print("Movies:", len(movie2encoded))

Encoding Created Successfully!
Users : 943
Movies: 1682


In [18]:
model = NCF(
    len(user2encoded),
    len(movie2encoded)
)

model.load_state_dict(
    torch.load(
        "models/ncf_model.pth",
        map_location=torch.device("cpu")
    )
)

model.eval()

print("NCF Model Loaded Successfully!")

NCF Model Loaded Successfully!


In [19]:
model = NCF(
    len(user2encoded),
    len(movie2encoded)
)

model.load_state_dict(
    torch.load(
        "models/ncf_model.pth",
        map_location=torch.device("cpu")
    )
)

model.eval()

print("NCF Model Loaded Successfully!")

NCF Model Loaded Successfully!


# Movie Recommendation Function

This function predicts ratings for all unseen movies of a selected user and returns the Top-N highest-rated recommendations.

In [20]:
def recommend_movies(user_id, top_n=10):

    # Check whether user exists
    if user_id not in user2encoded:
        print("User not found!")
        return None

    encoded_user = user2encoded[user_id]

    watched_movies = ratings[
        ratings.user_id == user_id
    ]["movie_id"].tolist()

    unseen_movies = [
        movie
        for movie in movie2encoded.keys()
        if movie not in watched_movies
    ]

    predictions = []

    for movie in unseen_movies:

        encoded_movie = movie2encoded[movie]

        user_tensor = torch.tensor([encoded_user])

        movie_tensor = torch.tensor([encoded_movie])

        with torch.no_grad():

            prediction = model(
                user_tensor,
                movie_tensor
            ).item()

        predictions.append(
            [movie, prediction]
        )

    prediction_df = pd.DataFrame(
        predictions,
        columns=[
            "movie_id",
            "predicted_rating"
        ]
    )

    prediction_df = prediction_df.merge(
        movies,
        on="movie_id"
    )

    prediction_df = prediction_df.sort_values(
        "predicted_rating",
        ascending=False
    )

    return prediction_df.head(top_n)

# Test Recommendation System

Generate movie recommendations for a sample user.

In [21]:
recommendations = recommend_movies(
    user_id=1,
    top_n=10
)

recommendations

,movie_id,predicted_rating,movie_title
103,483,4.803661,Casablanca (1942)
106,408,4.798072,"Close Shave, A (1995)"
987,1449,4.632718,Pather Panchali (1955)
499,313,4.579313,Titanic (1997)
984,1158,4.577496,"Fille seule, La (A Single Girl) (1995)"
322,736,4.508704,Shadowlands (1993)
502,493,4.503476,"Thin Man, The (1934)"
326,923,4.490463,Raise the Red Lantern (1991)
767,633,4.489809,"Christmas Carol, A (1938)"
418,316,4.464409,As Good As It Gets (1997)


# Display Recommendations

In [22]:
for i, row in recommendations.iterrows():

    print(f"{i+1}. {row['movie_title']}")
    print(f"Predicted Rating : {row['predicted_rating']:.3f}")
    print("-"*60)

104. Casablanca (1942)
Predicted Rating : 4.804
------------------------------------------------------------
107. Close Shave, A (1995)
Predicted Rating : 4.798
------------------------------------------------------------
988. Pather Panchali (1955)
Predicted Rating : 4.633
------------------------------------------------------------
500. Titanic (1997)
Predicted Rating : 4.579
------------------------------------------------------------
985. Fille seule, La (A Single Girl) (1995)
Predicted Rating : 4.577
------------------------------------------------------------
323. Shadowlands (1993)
Predicted Rating : 4.509
------------------------------------------------------------
503. Thin Man, The (1934)
Predicted Rating : 4.503
------------------------------------------------------------
327. Raise the Red Lantern (1991)
Predicted Rating : 4.490
------------------------------------------------------------
768. Christmas Carol, A (1938)
Predicted Rating : 4.490
------------------------------

# Notebook Summary

In this notebook, we:

- Loaded the trained NCF model.
- Created user and movie encoding dictionaries.
- Predicted ratings for unseen movies.
- Generated Top-N personalized recommendations.
- Verified that the recommendation pipeline works correctly.

The recommendation logic developed here will be reused directly in the final `app.py` Streamlit application.





# Predict Ratings for All Unseen Movies

In this section, we use the trained Neural Collaborative Filtering (NCF) model to predict ratings for every movie that the selected user has not yet watched.

The workflow includes:

- Identifying unseen movies
- Generating predicted ratings using the trained NCF model
- Ranking movies by predicted score
- Displaying the Top 10 personalized recommendations

This demonstrates how deep learning can personalize recommendations beyond traditional collaborative filtering approaches.

# Predict Ratings for All Unseen Movies

In this section, we use the trained Neural Collaborative Filtering (NCF) model to predict ratings for every movie that the selected user has not yet watched.

The workflow includes:

- Identifying unseen movies
- Generating predicted ratings using the trained NCF model
- Ranking movies by predicted score
- Displaying the Top 10 personalized recommendations

This demonstrates how deep learning can personalize recommendations beyond traditional collaborative filtering approaches.

In [24]:
print([x for x in globals().keys() if "recommend" in x.lower()])

['recommend_movies', 'recommendations']


In [25]:
print([x for x in globals().keys() if "top" in x.lower()])

[]


In [26]:
top10 = recommendations.head(10)

top10.to_csv(
    "models/sample_recommendations.csv",
    index=False
)

print("Recommendations Saved Successfully!")

Recommendations Saved Successfully!


In [29]:
print(recommendations.head())
print(recommendations.shape)

     movie_id  predicted_rating                             movie_title
103       483          4.803661                       Casablanca (1942)
106       408          4.798072                   Close Shave, A (1995)
987      1449          4.632718                  Pather Panchali (1955)
499       313          4.579313                          Titanic (1997)
984      1158          4.577496  Fille seule, La (A Single Girl) (1995)
(10, 3)


In [31]:
top10 = recommendations.head(10)

top10.to_csv(
    "models/sample_recommendations.csv",
    index=False
)

print("Recommendations Saved Successfully!")

Recommendations Saved Successfully!


In [32]:
import os

print(os.listdir("models"))

['item_mapping.csv', 'ncf_movie_embeddings.csv', 'sample_recommendations.csv', 'item_embeddings.csv', 'user_faiss.index', 'ncf_user_embeddings.csv', 'ncf_model.pth', 'user_mapping.csv', 'user_embeddings.csv']


In [33]:
# ==========================================
# Save Top Recommendations
# ==========================================

top10.to_csv(
    "models/sample_recommendations.csv",
    index=False
)

print("Recommendations Saved Successfully!")

Recommendations Saved Successfully!


In [34]:
import os

print(os.listdir("models"))

['item_mapping.csv', 'ncf_movie_embeddings.csv', 'sample_recommendations.csv', 'item_embeddings.csv', 'user_faiss.index', 'ncf_user_embeddings.csv', 'ncf_model.pth', 'user_mapping.csv', 'user_embeddings.csv']


# Notebook Summary

In this notebook, we successfully:

- Loaded the trained Neural Collaborative Filtering model
- Loaded the MovieLens dataset
- Encoded users and movies
- Generated personalized movie recommendations
- Ranked movies according to predicted ratings
- Displayed the Top 10 recommendations
- Saved sample recommendations for deployment

These recommendations will now be used inside the Streamlit application.